Quantization is a technique to downsize a trained model so that you can deploy it on EDGE devices.

In [1]:
import os
import tensorflow as tf
import tensorflow_model_optimization as tfmot
import numpy as np
from sklearn.metrics import accuracy_score

I0000 00:00:1794044582.111221    6306 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Model Training

In [2]:
(X_train, y_train) , (X_test, y_test) = tf.keras.datasets.mnist.load_data()

In [3]:
X_train = X_train.reshape(len(X_train), 28*28)
X_test = X_test.reshape(len(X_test), 28*28)

In [4]:
X_train = X_train /255
X_test = X_test / 255

In [5]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape = (784,)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation = "relu"),
    tf.keras.layers.Dense(10, activation = "softmax")
])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense (Dense)               (None, 100)               78500     
                                                                 
 dense_1 (Dense)             (None, 10)                1010      
                                                                 
Total params: 79510 (310.59 KB)
Trainable params: 79510 (310.59 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [6]:
model.compile(optimizer="adam", loss = "sparse_categorical_crossentropy",metrics=["accuracy"])

In [7]:
model.fit(X_train, y_train, epochs=5)

W0000 00:00:1794044669.573898    6306 cpu_allocator_impl.cc:82] Allocation of 188160000 exceeds 10% of free system memory.


Epoch 1/5
1875/1875 [==============================] - 11s 5ms/step - loss: 0.2698 - accuracy: 0.9233
Epoch 2/5
1875/1875 [==============================] - 9s 5ms/step - loss: 0.1209 - accuracy: 0.9650
Epoch 3/5
1875/1875 [==============================] - 9s 5ms/step - loss: 0.0849 - accuracy: 0.9753
Epoch 4/5
1875/1875 [==============================] - 9s 5ms/step - loss: 0.0649 - accuracy: 0.9805
Epoch 5/5
1875/1875 [==============================] - 9s 5ms/step - loss: 0.0502 - accuracy: 0.9846


In [8]:
tf_loss, tf_accuracy = model.evaluate(X_test,y_test)

W0000 00:00:1794044721.488280    6306 cpu_allocator_impl.cc:82] Allocation of 31360000 exceeds 10% of free system memory.


313/313 [==============================] - 1s 3ms/step - loss: 0.0802 - accuracy: 0.9742


In [9]:
model.export("./mnist_model/")

INFO:tensorflow:Assets written to: ./mnist_model/assets


INFO:tensorflow:Assets written to: ./mnist_model/assets


Saved artifact at './mnist_model/'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 784), dtype=tf.float32, name='input_1')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  135661028807200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135661028805968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135661028805440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135661028812128: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [10]:
model.save("./mnist_model.keras")

## (1) Post training quantization

### Without Quantization
**Convert into TFLite (FP32)**

In [11]:
converter = tf.lite.TFLiteConverter.from_saved_model("./mnist_model")
tflite_fp32_model = converter.convert()

W0000 00:00:1794044724.530130    6306 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794044724.530198    6306 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794044724.531037    6306 reader.cc:83] Reading SavedModel from: ./mnist_model
I0000 00:00:1794044724.532118    6306 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794044724.532144    6306 reader.cc:147] Reading SavedModel debug info (if present) from: ./mnist_model
I0000 00:00:1794044724.537705    6306 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1794044724.538912    6306 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794044724.573264    6306 loader.cc:220] Running initialization op on SavedModel bundle at path: ./mnist_model
I0000 00:00:1794044724.589298    6306 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 58290 microseconds.
I0000 00:00:1794044725.290158    6306 dump_mli

In [12]:
with open("tflite_fp32_model.tflite", "wb") as f:
    f.write(tflite_fp32_model)

### With Quantization

#### **TFLite Dynamic Range:** 
Original Model -> FP32 Weights -> Quantization -> INT8 Weights (where supported)

In [13]:
converter = tf.lite.TFLiteConverter.from_saved_model("./mnist_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_dynamic_quant_model = converter.convert()

W0000 00:00:1794044729.796767    6306 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794044729.796815    6306 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794044729.797159    6306 reader.cc:83] Reading SavedModel from: ./mnist_model
I0000 00:00:1794044729.797847    6306 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794044729.797859    6306 reader.cc:147] Reading SavedModel debug info (if present) from: ./mnist_model
I0000 00:00:1794044729.801389    6306 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794044729.823585    6306 loader.cc:220] Running initialization op on SavedModel bundle at path: ./mnist_model
I0000 00:00:1794044729.833907    6306 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 36778 microseconds.


In [14]:
with open("tflite_dynamic_quant_model.tflite", "wb") as f:
    f.write(tflite_dynamic_quant_model)

#### Full Integer Quantization (INT8)
Quantizing both weights and activations.

In [15]:
# Representative data to calibrate the model.
def representative_dataset():
    for i in range(100):
        sample = X_train[i:i+1].astype(np.float32)
        yield [sample]

In [16]:
converter = tf.lite.TFLiteConverter.from_saved_model("mnist_model")
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_int8_model = converter.convert()

W0000 00:00:1794044731.249430    6306 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794044731.249476    6306 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794044731.249840    6306 reader.cc:83] Reading SavedModel from: mnist_model
I0000 00:00:1794044731.250475    6306 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794044731.250489    6306 reader.cc:147] Reading SavedModel debug info (if present) from: mnist_model
I0000 00:00:1794044731.253878    6306 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794044731.277040    6306 loader.cc:220] Running initialization op on SavedModel bundle at path: mnist_model
I0000 00:00:1794044731.288513    6306 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 38696 microseconds.
fully_quantize: 0, inference_type: 6, input_inference_type: INT8, output_inference_type: INT8
W0000 00:00:1794044732.036897    6306 flatbuffer_export.cc:3851] Skipping 

In [17]:
with open("tflite_int8.tflite", "wb") as f:
    f.write(tflite_int8_model)

# (2) Quantization aware training

In [18]:
quantize_model = tfmot.quantization.keras.quantize_model
q_aware_model = quantize_model(model)

In [19]:
q_aware_model.compile(optimizer="adam", loss = "sparse_categorical_crossentropy",metrics=["accuracy"])

In [20]:
# For fine tunning quantization model 
# take a sub set
X_train_subset = X_train[:10000]
y_train_subset = y_train[:10000]

In [21]:
q_aware_model.fit(X_train_subset, y_train_subset, epochs=1)

313/313 [==============================] - 3s 5ms/step - loss: 0.0420 - accuracy: 0.9882


In [23]:
q_aware_loss, q_aware_accuracy = q_aware_model.evaluate(X_test, y_test, verbose=0)
print(f"Quantization aware model accuracy: {q_aware_accuracy}")

Quantization aware model accuracy: 0.9761999845504761


In [24]:
converter = tf.lite.TFLiteConverter.from_keras_model(q_aware_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant_aware_model = converter.convert()

INFO:tensorflow:Assets written to: /tmp/tmpcfy0yrrd/assets


INFO:tensorflow:Assets written to: /tmp/tmpcfy0yrrd/assets
/home/sureshsecond/venv/lib/python3.10/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1794045070.968048    6306 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1794045070.968090    6306 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1794045070.968448    6306 reader.cc:83] Reading SavedModel from: /tmp/tmpcfy0yrrd
I0000 00:00:1794045070.971321    6306 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1794045070.971343    6306 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpcfy0yrrd
I0000 00:00:1794045070.989006    6306 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1794045071.059379    6306 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpcfy0yrrd
I0000 00:00:1794045071

In [25]:
with open("tflite_quant_aware_model.tflite", "wb") as f:
    f.write(tflite_quant_aware_model)

### Run and Evaluate models

In [32]:
def evaluate_quantized_models(model_path, X_test, y_test):
    # Load model
    interpreter = tf.lite.Interpreter(model_path= model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_dtype = input_details["dtype"]
    input_index = input_details["index"]
    output_index = output_details["index"]
    
    # Get quantization parameters
    scale, zero_point = input_details["quantization"]
    
    correct = 0
    for i in range(len(X_test)):
        x = X_test[i:i+1].astype(np.float32)
        # Convert FP32 input to INT8 when required
        if np.issubdtype(input_dtype, np.integer):
            x = np.round(x / scale + zero_point)

            limits = np.iinfo(input_dtype)
            x = np.clip(x, limits.min, limits.max)

        x = x.astype(input_dtype)
        # Inference
        interpreter.set_tensor(input_index, x) # provides input
        interpreter.invoke() # executes the model.
        predictions = interpreter.get_tensor(output_index) # retrieves predictions.
        if y_test[i]==np.argmax(predictions):
            correct += 1
    accuracy = correct/ len(X_test)
    return accuracy

In [33]:
quant_fp32_accuracy = evaluate_quantized_models("./tflite_fp32_model.tflite",X_test, y_test)
dynamic_accuracy = evaluate_quantized_models("./tflite_dynamic_quant_model.tflite",X_test, y_test)
int8_accuracy = evaluate_quantized_models("./tflite_int8.tflite", X_test, y_test)
q_aware_accuracy = evaluate_quantized_models("./tflite_quant_aware_model.tflite", X_test, y_test)

print("Accuracy:-")
print(f"Tensorflow model accuracy: {tf_accuracy}")
print(f"TFLite FP32: {quant_fp32_accuracy}")
print(f"Dynamic Quantization: {dynamic_accuracy}")
print(f"TFLite INT8: {int8_accuracy}")
print(f"Tensorflow Quantization aware model accuracy: {q_aware_accuracy}")

/home/sureshsecond/venv/lib/python3.10/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Accuracy:-
Tensorflow model accuracy: 0.9742000102996826
TFLite FP32: 0.9742
Dynamic Quantization: 0.9745
TFLite INT8: 0.9741
Tensorflow Quantization aware model accuracy: 0.9762


### Compare size of the models

In [28]:
def get_model_size(export_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(export_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            # Skip if it's a symbolic link
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

In [29]:
export_path = "mnist_model"
size_bytes = get_model_size(export_path)
print(f"Tensorfloe Model Size in MB: {size_bytes / (1024 * 1024):.4f} MB")

tflite_fp32_model_size = os.path.getsize("./tflite_fp32_model.tflite")
tflite_fp32_model_size = tflite_fp32_model_size/ (1024*1024)
print(f"TFLite FP32 model Size: {tflite_fp32_model_size:.4f} MB")

tflite_dynamic_quant_model_size = os.path.getsize("./tflite_dynamic_quant_model.tflite")
tflite_dynamic_quant_model_size = tflite_dynamic_quant_model_size/ (1024*1024)
print(f"TFLite model Size: {tflite_dynamic_quant_model_size:.4f} MB")

tflite_quant_aware_model_size = os.path.getsize("./tflite_quant_aware_model.tflite")
tflite_quant_aware_model_size = tflite_quant_aware_model_size/ (1024*1024)
print(f"TFLite Quantization aware model Size: {tflite_quant_aware_model_size:.4f} MB")

Tensorfloe Model Size in MB: 0.3260 MB
TFLite FP32 model Size: 0.3052 MB
TFLite model Size: 0.0821 MB
TFLite Quantization aware model Size: 0.0789 MB
